# 🔍 Missing Data in Python & Pandas

---

## Why Missing Data Matters

In real-world data — whether from a bank's transaction logs, a survey, or a hospital database — data is almost never perfectly complete. Missing values are the norm, not the exception.

If you ignore missing data:
- **Calculations silently produce wrong results** (e.g., a loan average calculated over fewer records than expected)
- **Machine learning models fail or produce biased predictions**
- **Joins and merges create unexpected gaps**
- **Regulatory reports may be incorrect**

Understanding how to detect, count, and handle missing data is one of the most critical skills in data analysis.

---

## What You Will Learn

| Section | Topics |
|---|---|
| **Part 1: What is NaN** | Definition, three aliases, comparison rules |
| **Part 2: Where Missing Data Comes From** | Input data, merging, reindexing |
| **Part 3: Finding & Counting Missing Data** | `count()`, `isnull()`, `value_counts()`, numpy |
| **Part 4: Cleaning Missing Data** | Replace, forward fill, backward fill, interpolate, drop |
| **Part 5: Calculations with Missing Data** | Arithmetic behaviour, `skipna` parameter |

---

# Part 1: What is NaN?

---

## 1.1 The Concept of NaN

**NaN** stands for **Not a Number**. It is the standard representation of a **missing or undefined value** in Python's numerical ecosystem.

NaN is **not** the same as:

| Value | Looks like nothing? | Is NaN? | Why |
|---|---|---|---|
| `None` | Yes (Python null) | No | Python object, not a float |
| `0` | No | No | A valid number (zero) |
| `''` | Yes (empty string) | No | A string, not a number |
| `False` | Yes (falsy) | No | A boolean |
| `NaN` | Yes | ✅ Yes | The actual missing value marker |

---

## 1.2 Three Aliases — NaN, nan, NAN

NaN is defined in the **`numpy`** library (not in Python's built-in library).

NumPy provides **three aliases** that are all the same object:

| Alias | Case | Usage |
|---|---|---|
| `nan` | lowercase | Most common — used in code |
| `NaN` | mixed case | Used in display / documentation |
| `NAN` | uppercase | Less common |

All three are **identical** — they point to exactly the same float value: `float('nan')`.

In [ ]:
# Suppress deprecation and future warnings to keep output clean
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Import nan from numpy (the standard missing value constant)
# numpy 2.0 removed the NaN and NAN uppercase aliases — only 'nan' remains
from numpy import nan

# Create NaN and NAN as local aliases so all three names work in this notebook
# This mirrors the original numpy behaviour for teaching purposes
NaN = nan   # mixed-case alias — same object as nan
NAN = nan   # uppercase alias — same object as nan

# Confirm they are all the same Python object using the 'is' identity check
print("NaN is nan:",  NaN is nan)    # → True  (same object in memory)
print("NaN is NAN:",  NaN is NAN)    # → True
print("nan is NAN:",  nan is NAN)    # → True

# All three print as 'nan'
print("\nHow they display:")
print("NaN  →", NaN)
print("nan  →", nan)
print("NAN  →", NAN)

# All three have type 'float' — NaN is technically a floating-point value
print("\nType of NaN:", type(NaN))   # → <class 'float'>

---

## 1.3 The Most Unusual Property of NaN: It Is Not Equal to Anything

NaN has a uniquely counterintuitive behaviour:

> **NaN is not equal to anything — not even itself.**

This is defined in the IEEE 754 floating-point standard (the international standard for how computers handle decimal numbers).

This means you **cannot** use `==` to check if something is NaN. The comparison will always return `False`.

```python
NaN == NaN   # → False  ← This is surprising but correct!
```

This is why we need special functions (`pd.isnull()`, `pd.notnull()`) to detect NaN.

In [ ]:
# === Comparing NaN to other common values ===
# All of these return False because NaN is not equal to anything

# NaN is NOT True
print("NaN == True: ", NaN == True)    # → False

# NaN is NOT False
print("NaN == False:", NaN == False)   # → False

# NaN is NOT zero
print("NaN == 0:    ", NaN == 0)       # → False

# NaN is NOT an empty string
print("NaN == '':   ", NaN == '')      # → False

print()

# === Comparing NaN to other NaN values ===
# Even NaN is not equal to itself — this is the most surprising behaviour
print("NaN == NaN:",  NaN == NaN)     # → False  ← NaN ≠ itself!
print("NaN == nan:",  NaN == nan)     # → False
print("NaN == NAN:",  NaN == NAN)     # → False
print("nan == NAN:",  nan == NAN)     # → False

print()
print("KEY TAKEAWAY: Never use == to check for NaN. It always returns False.")

---

## 1.4 Testing for Missing Values the Right Way

Since `==` does not work, pandas provides two dedicated functions for NaN detection:

| Function | Returns `True` when | Returns `False` when |
|---|---|---|
| `pd.isnull(x)` | `x` is NaN (missing) | `x` has a real value |
| `pd.notnull(x)` | `x` has a real value | `x` is NaN (missing) |

These work on:
- **Scalar values** → `pd.isnull(nan)` → `True`
- **Series** → returns a boolean Series (one `True`/`False` per row)
- **DataFrames** → returns a boolean DataFrame (one `True`/`False` per cell)

**Equivalent methods:** `pd.isnull` = `pd.isna`, `pd.notnull` = `pd.notna`

In [ ]:
# Import pandas for the isnull/notnull functions
import pandas as pd

print("=== pd.isnull() — Returns True if the value IS missing ===")
# Testing all three NaN aliases — all correctly detected as missing
print("pd.isnull(NaN): ", pd.isnull(NaN))    # → True
print("pd.isnull(nan): ", pd.isnull(nan))    # → True
print("pd.isnull(NAN): ", pd.isnull(NAN))    # → True

print()
print("=== pd.notnull() — Returns True if the value is NOT missing ===")
# NaN is missing, so notnull returns False
print("pd.notnull(NaN):       ", pd.notnull(NaN))         # → False
# A real number is not missing, so notnull returns True
print("pd.notnull(42):        ", pd.notnull(42))           # → True
# A string is not missing, so notnull returns True
print("pd.notnull('hello'):   ", pd.notnull('hello'))      # → True
# Zero is a real value, not missing
print("pd.notnull(0):         ", pd.notnull(0))            # → True
# None (Python null) IS treated as missing by pandas
print("pd.isnull(None):       ", pd.isnull(None))          # → True

print()
print("=== Using on a list or Series ===")
# When applied to a Series, isnull returns a boolean Series — True where NaN exists
sample = pd.Series([1.0, nan, 3.0, nan, 5.0])
print("Series:          ", sample.tolist())
print("isnull mask:     ", pd.isnull(sample).tolist())    # [False, True, False, True, False]
print("notnull mask:    ", pd.notnull(sample).tolist())   # [True, False, True, False, True]

---

# Part 2: Where Does Missing Data Come From?

---

## Overview

Missing data can enter your DataFrame from several sources:

| Source | When it happens |
|---|---|
| **1. Input data** | The raw CSV/Excel/database already has blank cells |
| **2. Merging data** | Two DataFrames joined together — some rows have no match |
| **3. Re-indexing** | You request rows/columns that don't exist in the original data |
| **4. User code** | Manually assigning `nan`, or bugs in transformation logic |

Understanding the **source** of missing data helps you choose the right strategy to handle it.

---

## 2.1 Source 1 — Missing Data from Input Files

When `pd.read_csv()` reads a CSV file, it automatically converts certain cell values into `NaN`.

**Default values treated as NaN by pandas:**
`''`, `'NA'`, `'N/A'`, `'NaN'`, `'nan'`, `'null'`, `'NULL'`, `'None'`, `'n/a'`, `'#N/A'`, `'-'`, etc.

You can control this behaviour with two `read_csv()` parameters:

| Parameter | Effect |
|---|---|
| `keep_default_na=True` (default) | Pandas automatically converts standard missing-value strings to NaN |
| `keep_default_na=False` | Disable automatic conversion — you see raw strings |
| `na_values=[...]` | Manually specify additional strings to treat as NaN |

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

# === Build a small sample CSV in memory to simulate a file with blank cells ===
# In real code you would use: pd.read_csv('path/to/survey_visited.csv')
# We simulate it here so the notebook is self-contained

# This CSV has two 'dated' cells that are intentionally blank (simulating empty fields)
visited_data = """ident,site,dated
619,DR-1,1927-02-08
622,DR-1,
734,DR-3,1939-01-07
735,DR-3,
751,DR-3,1930-02-26
752,DR-3,-
837,MSK-4,1932-01-14
844,DR-1,1932-03-22"""

print("=" * 55)
print("LOAD 1: Default settings (keep_default_na=True)")
print("=" * 55)
# Default: pandas automatically replaces blank cells with NaN
df_default = pd.read_csv(StringIO(visited_data))
print(df_default.to_string())
print("\ndtypes:\n", df_default.dtypes)
print("\nNotice: blank cells and '-' are loaded as NaN automatically")

In [ ]:
# Re-create the sample data string for re-use
visited_data = """ident,site,dated
619,DR-1,1927-02-08
622,DR-1,
734,DR-3,1939-01-07
735,DR-3,
751,DR-3,1930-02-26
752,DR-3,-
837,MSK-4,1932-01-14
844,DR-1,1932-03-22"""

print("=" * 55)
print("LOAD 2: keep_default_na=False (raw strings preserved)")
print("=" * 55)
# keep_default_na=False: turns off automatic NaN conversion
# Blank cells are read as empty strings ''; '-' stays as '-'
# Use this when you want to see EXACTLY what is in your file
df_raw = pd.read_csv(StringIO(visited_data), keep_default_na=False)
print(df_raw.to_string())
print("\nNotice: blank cells appear as '' (empty string), not NaN")

In [ ]:
# Re-create the sample data string
visited_data = """ident,site,dated
619,DR-1,1927-02-08
622,DR-1,
734,DR-3,1939-01-07
735,DR-3,
751,DR-3,1930-02-26
752,DR-3,-
837,MSK-4,1932-01-14
844,DR-1,1932-03-22"""

print("=" * 55)
print("LOAD 3: Manual na_values=['', '-']")
print("=" * 55)
# na_values=['', '-']: tell pandas EXACTLY which strings should become NaN
# keep_default_na=False: disable built-in defaults so only our list is used
# This gives precise control over what counts as 'missing'
df_manual = pd.read_csv(StringIO(visited_data), na_values=['', '-'], keep_default_na=False)
print(df_manual.to_string())
print("\nNotice: both empty cells AND '-' are now NaN")
print("\nMissing count per column:")
print(df_manual.isnull().sum())

---

## 2.2 Source 2 — Missing Data from Merging

When you **merge (join)** two DataFrames, rows that don't find a match in the other table get filled with `NaN`.

This mirrors SQL JOIN behaviour:

| Join type | Missing data behaviour |
|---|---|
| **Inner join** (default) | Only keeps matching rows — no NaN from unmatched rows |
| **Left join** | Keeps ALL left rows; unmatched right-side values become NaN |
| **Right join** | Keeps ALL right rows; unmatched left-side values become NaN |
| **Outer join** | Keeps ALL rows from both; any unmatched values become NaN |

In [ ]:
# Build two small DataFrames that share some IDs but not all

# DataFrame 1: site visits — has site IDs 101, 102, 103, 104
df_visited = pd.DataFrame({
    'ident': [101, 102, 103, 104],
    'site':  ['DR-1', 'DR-2', 'DR-3', 'DR-4'],
    'dated': ['2020-01-01', '2020-01-05', '2020-01-10', '2020-01-15']
})

# DataFrame 2: measurements — only has records for site IDs 101, 102, 103
# ID 104 has no measurements (will cause NaN when merged)
df_survey = pd.DataFrame({
    'taken':    [101, 101, 102, 103],
    'person':   ['dyer', 'dyer', 'lake', 'roe'],
    'quant':    ['rad', 'sal', 'sal', 'sal'],
    'reading':  [9.82, 0.13, 0.09, 41.6]
})

print("=== Left DataFrame (visited sites) ===")
print(df_visited.to_string(), "\n")

print("=== Right DataFrame (survey measurements) ===")
print(df_survey.to_string(), "\n")

# Merge the two DataFrames on the shared ID column
# how='left': keep ALL rows from df_visited, even if no match in df_survey
# left_on='ident': the key column in the left DataFrame
# right_on='taken': the matching key column in the right DataFrame
df_merged_left = df_visited.merge(right=df_survey, left_on='ident', right_on='taken', how='left')

print("=== After LEFT MERGE ===")
print(df_merged_left.to_string())
print("\nNotice: Site DR-4 (ident=104) has no survey data → NaN in person, quant, reading, taken")
print("\nMissing values per column after merge:")
print(df_merged_left.isnull().sum())

---

## 2.3 Source 3 — Missing Data from Re-indexing

**Re-indexing** means asking pandas to reshape a Series or DataFrame to match a new set of index labels.

If the new index contains labels that didn't exist in the original, pandas fills those positions with `NaN`.

**Common scenario:**  
You have time series data for specific years (e.g., 2002, 2007) and want to expand it to cover every year from 2000 to 2010. The years with no data get `NaN`.

In [ ]:
# Build a small time series with gaps — like a gapminder life expectancy summary
# Data only exists for specific survey years (every 5 years)
life_exp = pd.Series(
    data =  [64.98, 66.10, 67.35, 68.68, 70.06, 71.19],
    index = [1987,  1992,  1997,  2002,  2007,  2012],
    name  = 'mean_lifeExp'
)

print("=== Original Series (data only at survey years) ===")
print(life_exp)
print()

# Re-index to cover EVERY year from 2000 to 2012
# range(2000, 2013) = [2000, 2001, 2002, ..., 2012]
# Years 2000, 2001, 2003-2006, 2008-2011 have no data → filled with NaN
reindexed = life_exp.reindex(range(2000, 2013))

print("=== After reindex(range(2000, 2013)) ===")
print(reindexed)
print()
print("Missing values introduced by re-indexing:", reindexed.isnull().sum())
print("Reason: Years 2000, 2001, 2003-2006, 2008-2011 did not exist in original Series")

---

# Part 3: Finding and Counting Missing Data

---

## Overview of Detection Methods

Before deciding how to handle missing data, you need to **quantify** it:
- How many values are missing in total?
- Which columns have missing values?
- What percentage of a column is missing?

| Method | What it does |
|---|---|
| `df.count()` | Count of **non-missing** values per column |
| `df.shape[0] - df.count()` | Number of **missing** values per column |
| `df.isnull().sum()` | Same as above — cleaner syntax |
| `np.count_nonzero(df.isnull())` | Total missing values across the whole DataFrame |
| `df['col'].value_counts(dropna=False)` | Frequency table including NaN count |

In [ ]:
# Build a representative dataset with missing values across several columns
# This simulates a real dataset like the Ebola country timeseries
country_data = {
    'Date':              ['2014-12-04', '2014-12-03', '2014-12-02', '2014-11-29',
                          '2014-11-28', '2014-11-26', '2014-11-22', '2014-11-18'],
    'Cases_Guinea':      [2430, np.nan, 2357, 2314, np.nan, 2260, 2197, np.nan],
    'Cases_Liberia':     [np.nan, 7069, np.nan, 6900, np.nan, 6669, np.nan, 6525],
    'Cases_SierraLeone': [6190, 6143, np.nan, 5942, 5765, np.nan, 5586, 5447],
    'Deaths_Guinea':     [1460, np.nan, 1437, 1408, np.nan, 1327, np.nan, 1262]
}

# Create the DataFrame from our simulated data dictionary
df_countries = pd.DataFrame(country_data)

print("=== Full Dataset ===")
print(df_countries.to_string())
print(f"\nShape: {df_countries.shape[0]} rows × {df_countries.shape[1]} columns")

In [ ]:
print("=" * 50)
print("METHOD 1: df.count() — Non-missing count per column")
print("=" * 50)
# .count() counts only rows where the value is NOT null
# Compare each column's count to the total rows (8) to see how many are missing
print(df_countries.count())
print("\nTotal rows:", df_countries.shape[0])
print("(A column with count < 8 has missing values)")

In [ ]:
print("=" * 55)
print("METHOD 2: Total rows minus count = missing count")
print("=" * 55)
# Get the total number of rows in the DataFrame
num_rows = df_countries.shape[0]

# Subtract non-missing count from total rows to get missing count per column
num_missing = num_rows - df_countries.count()
print("Missing values per column:")
print(num_missing)

print()
print("=" * 55)
print("METHOD 2b: Cleaner — isnull().sum()")
print("=" * 55)
# This is the most common and readable approach
# isnull() creates a boolean DataFrame (True=missing, False=present)
# .sum() adds up the Trues (True=1, False=0) to get a count
print(df_countries.isnull().sum())

print()
# Calculate what percentage of each column is missing
pct_missing = (df_countries.isnull().sum() / len(df_countries) * 100).round(1)
print("Percentage missing per column:")
print(pct_missing)

In [ ]:
import numpy as np

print("=" * 55)
print("METHOD 3: numpy.count_nonzero on isnull mask")
print("=" * 55)
# isnull() returns a boolean DataFrame
# np.count_nonzero counts all True values (True = missing) across the entire DataFrame
# This gives ONE number: total missing cells in the whole DataFrame
total_missing = np.count_nonzero(df_countries.isnull())
print(f"Total missing cells in the entire DataFrame: {total_missing}")

# Apply to a single column for a column-specific total
guinea_missing = np.count_nonzero(df_countries['Cases_Guinea'].isnull())
print(f"Missing in Cases_Guinea column only: {guinea_missing}")

In [ ]:
print("=" * 55)
print("METHOD 4: value_counts() with dropna parameter")
print("=" * 55)

# Default value_counts() EXCLUDES NaN from the frequency table
print("value_counts() — NaN excluded (default):")
print(df_countries['Cases_Guinea'].value_counts().head())

print()

# With dropna=False, NaN IS included as its own row in the frequency table
# This lets you see 'how many times is a value missing' directly
print("value_counts(dropna=False) — NaN included:")
print(df_countries['Cases_Guinea'].value_counts(dropna=False).head())

---

# Part 4: Cleaning Missing Data — 5 Strategies

---

## When to Use Which Strategy

There is no universal best approach. The right strategy depends on:
- **What the data represents** (time series? categorical? numeric?)
- **Why the data is missing** (random? systematic? structural?)
- **How much data is missing** (1% vs 50%?)

| Strategy | Method | Best For |
|---|---|---|
| **Replace with constant** | `fillna(value)` | Replace with 0, mean, or a known default |
| **Forward fill** | `ffill()` | Time series — carry the last known value forward |
| **Backward fill** | `bfill()` | Time series — use the next known value |
| **Interpolation** | `interpolate()` | Smoothly estimate values between known points |
| **Drop rows/columns** | `dropna()` | When missing rows are genuinely unusable |

---

## Strategy 1: Replace with a Constant — `fillna(value)`

The simplest approach: replace every `NaN` with a fixed value (e.g., 0, the mean, a placeholder string).

In [ ]:
# Show only the first 5 rows and first 4 columns to keep output manageable
print("=== BEFORE fillna: Original data (NaN values visible) ===")
print(df_countries.iloc[0:5, 0:4].to_string())

print()
print("=== AFTER fillna(0): All NaN replaced with zero ===")
# fillna(0): replace every NaN in the entire DataFrame with the value 0
# This does NOT modify df_countries — it returns a NEW DataFrame
# Useful when NaN genuinely means 'zero cases' or 'zero amount'
print(df_countries.fillna(0).iloc[0:5, 0:4].to_string())

print()
print("=== fillna with column mean (smarter default) ===")
# A more statistically sound approach: fill with the column's own mean
# This preserves the overall distribution better than using 0
df_mean_filled = df_countries.copy()
for col in ['Cases_Guinea', 'Cases_Liberia', 'Cases_SierraLeone']:
    # Fill NaN in each numeric column with that column's mean value
    col_mean = df_countries[col].mean()
    df_mean_filled[col] = df_mean_filled[col].fillna(col_mean)
print(df_mean_filled.iloc[0:5, 0:4].round(0).to_string())

---

## Strategy 2: Forward Fill — `ffill()`

**Forward fill** propagates the **last valid value** downward to fill the gaps below it.

```
Before:    1.0  →  After ffill:  1.0
           NaN  →                1.0   ← filled from above
           NaN  →                1.0   ← filled from above
           4.0  →                4.0
           NaN  →                4.0   ← filled from above
```

**Best for:** Time series where values don't change between observation points (e.g., sensor readings, stock prices when the market is closed).

⚠️ **Caution:** If the FIRST row is NaN, forward fill cannot fill it (nothing above to copy from).

In [ ]:
print("=== BEFORE: Original (notice NaN pattern) ===")
print(df_countries.iloc[0:6, 0:4].to_string())

print()
print("=== AFTER ffill(): Each NaN takes the value from the row ABOVE it ===")
# ffill() = 'forward fill' — propagate last valid value downward
# Also written as: fillna(method='ffill') in older pandas versions
df_ffilled = df_countries.ffill()
print(df_ffilled.iloc[0:6, 0:4].to_string())
print()
print("Remaining NaN after ffill (only possible in first row or all-NaN columns):")
print(df_ffilled.isnull().sum())

---

## Strategy 3: Backward Fill — `bfill()`

**Backward fill** propagates the **next valid value** upward to fill the gaps above it — the opposite direction of forward fill.

```
Before:    NaN  →  After bfill:  1.0   ← filled from below
           NaN  →                1.0   ← filled from below
           1.0  →                1.0
           NaN  →                4.0   ← filled from below
           4.0  →                4.0
```

**Best for:** When you know the next future value is the best estimate for the gap (e.g., filling in a forecast that starts later).

⚠️ **Caution:** If the LAST row is NaN, backward fill cannot fill it.

In [ ]:
print("=== BEFORE: Original ===")
print(df_countries.iloc[0:6, 0:4].to_string())

print()
print("=== AFTER bfill(): Each NaN takes the value from the row BELOW it ===")
# bfill() = 'backward fill' — propagate next valid value upward
# Also written as: fillna(method='bfill') in older pandas versions
df_bfilled = df_countries.bfill()
print(df_bfilled.iloc[0:6, 0:4].to_string())
print()

# Side-by-side comparison for one column
print("=== Side-by-side: Original vs ffill vs bfill (Cases_Guinea) ===")
comparison = pd.DataFrame({
    'Original':    df_countries['Cases_Guinea'],
    'Forward_Fill': df_countries['Cases_Guinea'].ffill(),
    'Backward_Fill': df_countries['Cases_Guinea'].bfill()
})
print(comparison.to_string())

---

## Strategy 4: Interpolation — `interpolate()`

**Interpolation** estimates missing values by **calculating what would make sense between known values**.

The default method is **linear interpolation** — it draws a straight line between the two nearest known values and places the missing values evenly along that line.

```
Before:    100.0  →  After interpolate:  100.0
           NaN    →                      150.0   ← midpoint between 100 and 200
           200.0  →                      200.0
           NaN    →                      225.0   ← midpoint between 200 and 250
           NaN    →                      237.5
           250.0  →                      250.0
```

**Best for:** Numeric time series where a smooth progression between values is expected (e.g., population, temperature, prices).

⚠️ **Caution:** Cannot fill NaN at the very beginning or end of a series (no boundary point on one side).

In [ ]:
print("=== BEFORE: Original ===")
print(df_countries.iloc[0:6, 0:4].to_string())

print()
print("=== AFTER interpolate(): Gaps filled with linearly estimated values ===")
# interpolate() fills NaN with values estimated by linear interpolation between known points
# method='linear' (default): assumes equal spacing, draws straight line between known values
# Note: interpolate() only works on NUMERIC columns — string/object columns are skipped
numeric_cols = df_countries.select_dtypes(include='number').columns
df_interpolated = df_countries.copy()
df_interpolated[numeric_cols] = df_countries[numeric_cols].interpolate(method='linear')
print(df_interpolated.iloc[0:6, 0:4].to_string())

print()
print("=== Detailed look at Cases_Guinea: Original → Interpolated ===")
guinea_comparison = pd.DataFrame({
    'Original':      df_countries['Cases_Guinea'],
    'Interpolated':  df_countries['Cases_Guinea'].interpolate()
})
print(guinea_comparison.to_string())
print("\nNotice: NaN at row 1 was filled with the midpoint between rows 0 and 2")

---

## Strategy 5: Drop Missing Values — `dropna()`

`dropna()` removes rows (or columns) that contain any missing values.

Use this when:
- You have **enough data** that losing some rows is acceptable
- Rows with missing values are **not useful** for your analysis
- The data is missing for a **systematic reason** that makes those rows unreliable

**Key parameters:**

| Parameter | Options | Effect |
|---|---|---|
| `axis` | `0` (rows, default), `1` (columns) | Drop rows or drop columns |
| `how` | `'any'` (default), `'all'` | Drop if ANY cell is NaN, or only if ALL cells are NaN |
| `subset` | list of column names | Only check specific columns for NaN |
| `thresh` | integer | Keep rows with at least N non-NaN values |

In [ ]:
print("=== Shape BEFORE dropna ===")
# df.shape returns (rows, columns) — a tuple
print(f"Rows: {df_countries.shape[0]}, Columns: {df_countries.shape[1]}")

print()
print("=== Shape AFTER dropna() — default: drops ANY row with a NaN ===")
# dropna() removes every row that has at least one NaN cell
# how='any' is the default: any NaN in the row → row is dropped
df_dropped = df_countries.dropna()
print(f"Rows: {df_dropped.shape[0]}, Columns: {df_dropped.shape[1]}")
print(df_dropped.to_string())

print()
print("=== dropna(how='all') — only drops rows where EVERY cell is NaN ===")
# Much less aggressive — a row must be completely empty to be removed
df_dropped_all = df_countries.dropna(how='all')
print(f"Rows after dropna(how='all'): {df_dropped_all.shape[0]}")

print()
print("=== dropna(subset=['Cases_Guinea']) — only check one column ===")
# Only drop rows where Cases_Guinea is NaN (other columns' NaN don't matter)
df_dropped_subset = df_countries.dropna(subset=['Cases_Guinea'])
print(f"Rows remaining: {df_dropped_subset.shape[0]}")
print(df_dropped_subset[['Date','Cases_Guinea']].to_string())

---

# Part 5: Calculations with Missing Data

---

## 5.1 Arithmetic with NaN — NaN is Contagious

When you perform arithmetic operations (add, subtract, multiply) between columns, and **any** of the input values is NaN, the result is also NaN.

This is called **NaN propagation** — NaN is 'contagious' in arithmetic.

```python
1000 + NaN + 500  →  NaN    (not 1500!)
```

This is mathematically correct: if one input is unknown, the sum is also unknown. But it can silently produce unexpected results in your analysis.

In [ ]:
# Demonstrate NaN propagation in column arithmetic

# Add three columns together — any row where ANY column has NaN results in NaN total
# This is the standard arithmetic behaviour: NaN + anything = NaN
df_countries['Multiple'] = (df_countries['Cases_Guinea'] +
                             df_countries['Cases_Liberia'] +
                             df_countries['Cases_SierraLeone'])

# Show only the relevant columns
cols = ['Cases_Guinea', 'Cases_Liberia', 'Cases_SierraLeone', 'Multiple']
print("=== Column arithmetic: NaN propagation ===")
print(df_countries[cols].to_string())
print()
print("KEY OBSERVATION: 'Multiple' is only a real number when ALL three source")
print("columns have data. If even ONE is NaN, the sum is NaN.")
print()
print("Missing in 'Multiple' column:", df_countries['Multiple'].isnull().sum())

---

## 5.2 Aggregations with NaN — The `skipna` Parameter

Built-in aggregation functions like `sum()`, `mean()`, `min()`, `max()`, `std()` have a **`skipna`** parameter that controls how they handle NaN:

| `skipna` | Behaviour | When to Use |
|---|---|---|
| `True` (default) | NaN values are **ignored** in the calculation | Most analysis — produce useful results despite missing data |
| `False` | If ANY NaN present, result is **NaN** | When you need to know if data is complete before trusting the result |

In [ ]:
# Isolate the three case columns for clear demonstration
cols = ['Cases_Guinea', 'Cases_Liberia', 'Cases_SierraLeone', 'Multiple']

print("=" * 60)
print("sum() — Default (skipna=True): NaN values are IGNORED")
print("=" * 60)
# skipna=True means NaN cells are simply skipped in the addition
# So sum of [2430, NaN, 2357, NaN] = 2430 + 2357 = 4787 (NaN rows skipped)
print(df_countries[cols].sum())

print()
print("=" * 60)
print("sum(skipna=True) — Explicit (same as default)")
print("=" * 60)
# Explicitly specifying skipna=True — same as the default above
print(df_countries[cols].sum(skipna=True))

print()
print("=" * 60)
print("sum(skipna=False) — ANY NaN in column makes sum = NaN")
print("=" * 60)
# skipna=False: if ANY NaN exists in a column, the entire sum is NaN
# This is strict mode: tells you a column is incomplete before trusting its total
print(df_countries[cols].sum(skipna=False))
print()
print("Explanation:")
print("  Cases_Guinea has NaN → sum is NaN (even though some values exist)")
print("  Cases_Liberia has NaN → sum is NaN")
print("  Use skipna=False as a DATA QUALITY CHECK before reporting totals")

In [ ]:
# skipna applies to all major aggregation functions — not just sum()
guinea = df_countries['Cases_Guinea']

print("=== skipna works the same way for mean, min, max, std ===")
print(f"mean (skipna=True):  {guinea.mean(skipna=True):.2f}")
print(f"mean (skipna=False): {guinea.mean(skipna=False)}")

print()
print(f"min  (skipna=True):  {guinea.min(skipna=True)}")
print(f"min  (skipna=False): {guinea.min(skipna=False)}")

print()
print(f"max  (skipna=True):  {guinea.max(skipna=True)}")
print(f"max  (skipna=False): {guinea.max(skipna=False)}")

print()
print("Non-NaN count:", guinea.count())
print("Total rows:   ", len(guinea))
print("NaN count:    ", guinea.isnull().sum())

---

# Summary

---

## Key Concepts at a Glance

### Part 1: What is NaN

| Concept | Key Point |
|---|---|
| NaN definition | Not a Number — the standard missing value marker |
| Three aliases | `nan`, `NaN`, `NAN` — all the same float object from numpy |
| Comparison rule | `NaN == NaN` → **False** — NaN is not equal to anything |
| Correct detection | Use `pd.isnull()` or `pd.notnull()`, never `==` |

### Part 2: Sources of Missing Data

| Source | Cause |
|---|---|
| Input data | Blank cells in CSV/Excel read as NaN |
| Merging | Unmatched rows in a join get NaN |
| Re-indexing | New index labels with no data → NaN |

### Part 3: Detecting Missing Data

| Method | Use |
|---|---|
| `df.isnull().sum()` | Missing count per column |
| `np.count_nonzero(df.isnull())` | Total missing cells in DataFrame |
| `value_counts(dropna=False)` | Frequency table including NaN |

### Part 4: Cleaning Strategies

| Method | Code | Best For |
|---|---|---|
| Replace | `df.fillna(0)` | Known default values |
| Forward fill | `df.ffill()` | Carry last value forward |
| Backward fill | `df.bfill()` | Use next value |
| Interpolate | `df.interpolate()` | Smooth numeric time series |
| Drop | `df.dropna()` | Remove incomplete rows |

### Part 5: Calculations

| Rule | Example |
|---|---|
| NaN is contagious in arithmetic | `1000 + NaN = NaN` |
| `skipna=True` (default) | NaN is ignored — calculation proceeds |
| `skipna=False` | Any NaN → entire result is NaN |

---

## Self-Test Questions

1. Why does `NaN == NaN` return `False`? How do you correctly check if a value is NaN?
2. What are the three aliases for NaN in numpy? Are they the same object?
3. Name three common sources of missing data in pandas DataFrames.
4. What is the difference between `dropna(how='any')` and `dropna(how='all')`?
5. When would you use `ffill()` vs `bfill()` vs `interpolate()`?
6. What does `skipna=False` do in `df.sum()`? When is this useful?